# TalabaBot: transformer modelini oʻqitish (FR-31)

1. Kompyuterda `python -m experiments.train_baselines` ni ishga tushiring: `data/transformer_input.json` hosil boʻladi.
2. Colab'da **Runtime → Change runtime type → GPU** ni tanlang.
3. **Runtime → Run all** ni bosing va soʻralganda `transformer_input.json` ni yuklang.
4. Oxirida `transformer_output.zip` yuklanadi. Undagi `transformer/` papkani loyihadagi `models/` ichiga, `transformer.json` ni `experiments/results/` ichiga qoʻying.

Matnlar kompyuterda maskalangan, baribir Colab'ga faqat oʻzingizning Google akkauntingizdan yuklang.

In [ ]:
# colab-only
!pip install -q transformers==5.22.2 sentencepiece==0.2.3
from google.colab import files

files.upload()

In [ ]:
import json
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = os.environ.get("MODEL_NAME", "xlm-roberta-base")
EPOCHS = int(os.environ.get("EPOCHS", 4))
INPUT_PATH = Path(os.environ.get("INPUT_PATH", "transformer_input.json"))
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "transformer_output"))
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_LENGTH = 128
SEED = 42
CPU_TIMING_SAMPLES = 100
DIGITS = 6

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Qurilma:", device)
data = json.loads(INPUT_PATH.read_text(encoding="utf-8"))
labels = data["labels"]
label2id = {label: index for index, label in enumerate(labels)}
id2label = {index: label for label, index in label2id.items()}
train_rows, test_rows = data["train"], data["test"]
print(len(train_rows), "train,", len(test_rows), "test")

In [ ]:
def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


def train_model(field):
    set_seed()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(device)

    targets = [label2id[row["label"]] for row in train_rows]
    weights = compute_class_weight("balanced", classes=np.arange(len(labels)), y=targets)
    loss_function = torch.nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float, device=device)
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    def collate(batch):
        encoded = tokenizer(
            [row[field] for row in batch],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        )
        return encoded, torch.tensor([label2id[row["label"]] for row in batch])

    loader = DataLoader(
        train_rows,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate,
        generator=torch.Generator().manual_seed(SEED),
    )
    model.train()
    for epoch in range(EPOCHS):
        total = 0.0
        for encoded, target in loader:
            optimizer.zero_grad()
            logits = model(**encoded.to(device)).logits
            loss = loss_function(logits, target.to(device))
            loss.backward()
            optimizer.step()
            total += loss.item()
        print(f"{field}, davr {epoch + 1}: loss = {total / len(loader):.4f}")
    return model.eval(), tokenizer


def predict(model, tokenizer, texts):
    result = []
    with torch.no_grad():
        for start in range(0, len(texts), BATCH_SIZE):
            encoded = tokenizer(
                texts[start : start + BATCH_SIZE],
                truncation=True,
                max_length=MAX_LENGTH,
                padding=True,
                return_tensors="pt",
            ).to(next(model.parameters()).device)
            result.extend(model(**encoded).logits.argmax(-1).tolist())
    return [id2label[index] for index in result]


def rounded(value):
    return round(float(value), DIGITS)


def scores(truth, predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(
        truth, predicted, average="macro", zero_division=0
    )
    return {
        "accuracy": rounded(accuracy_score(truth, predicted)),
        "macro_precision": rounded(precision),
        "macro_recall": rounded(recall),
        "macro_f1": rounded(f1),
    }


def holdout(truth, predicted):
    report = classification_report(
        truth, predicted, labels=labels, output_dict=True, zero_division=0
    )
    per_category = {
        label: {key: rounded(value) for key, value in report[label].items()} for label in labels
    }
    matrix = confusion_matrix(truth, predicted, labels=labels).tolist()
    return scores(truth, predicted) | {"per_category": per_category, "confusion_matrix": matrix}


def robustness(normalized, raw):
    result = {}
    for subset in ("cyrillic", "typo"):
        members = [row for row in test_rows if row[subset]]
        entry = {"n": len(members)}
        if members:
            truth = [row["label"] for row in members]
            entry["normalized"] = scores(
                truth, predict(*normalized, [row["norm"] for row in members])
            )
            entry["raw"] = scores(truth, predict(*raw, [row["raw"] for row in members]))
        result[subset] = entry
    return result


def mean_cpu_ms(model, tokenizer, texts):
    model = model.to("cpu")
    started = time.perf_counter()
    for text in texts:
        predict(model, tokenizer, [text])
    return (time.perf_counter() - started) * 1000 / len(texts)

In [ ]:
normalized = train_model("norm")
raw = train_model("raw")

truth = [row["label"] for row in test_rows]
result = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "epochs": EPOCHS,
    "holdout": holdout(truth, predict(*normalized, [row["norm"] for row in test_rows])),
    "cv": None,
    "robustness": robustness(normalized, raw),
}
result["mean_prediction_ms_cpu"] = mean_cpu_ms(
    *normalized, [row["norm"] for row in test_rows[:CPU_TIMING_SAMPLES]]
)
print("macro-F1:", result["holdout"]["macro_f1"])
print("CPU, ms:", round(result["mean_prediction_ms_cpu"], 1))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
normalized[0].save_pretrained(OUTPUT_DIR / "transformer")
normalized[1].save_pretrained(OUTPUT_DIR / "transformer")
(OUTPUT_DIR / "transformer.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
)
archive = shutil.make_archive("transformer_output", "zip", OUTPUT_DIR)
print("Tayyor:", archive)

In [ ]:
# colab-only
files.download("transformer_output.zip")